In [2]:
import pandas as pd
from pathlib import Path
from tqdm import tqdm

pd.set_option("display.max_columns", 200)

In [3]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[1] 
DATA_DIR = PROJECT_ROOT / "data"

In [4]:


BRONZE_LAPS_BASE = PROJECT_ROOT / "data/bronze/laps/season=2024"
SILVER_LAPS_BASE = PROJECT_ROOT / "data/silver/fact_laps/season=2024"


In [5]:
rounds = sorted([p for p in BRONZE_LAPS_BASE.iterdir() if p.is_dir()])
len(rounds), rounds[:3], rounds[-3:]


(24,
 [PosixPath('/Users/poojithanappari/Desktop/github/F1-ENE-datapipeline-project/data/bronze/laps/season=2024/round=1'),
  PosixPath('/Users/poojithanappari/Desktop/github/F1-ENE-datapipeline-project/data/bronze/laps/season=2024/round=10'),
  PosixPath('/Users/poojithanappari/Desktop/github/F1-ENE-datapipeline-project/data/bronze/laps/season=2024/round=11')],
 [PosixPath('/Users/poojithanappari/Desktop/github/F1-ENE-datapipeline-project/data/bronze/laps/season=2024/round=7'),
  PosixPath('/Users/poojithanappari/Desktop/github/F1-ENE-datapipeline-project/data/bronze/laps/season=2024/round=8'),
  PosixPath('/Users/poojithanappari/Desktop/github/F1-ENE-datapipeline-project/data/bronze/laps/season=2024/round=9')])

In [14]:
def transform_bronze_laps_to_silver(df: pd.DataFrame) -> pd.DataFrame:
    silver = pd.DataFrame()

    silver["season"] = df["season"].astype("int64")
    silver["event_round"] = df["event_round"].astype("int64")
    silver["session_type"] = df["session_type"].astype("string")

    silver["Driver"] = df["Driver"].astype("string")
    silver["LapNumber"] = df["LapNumber"].astype("int64")

    # Timedelta → seconds (this is correct and intentional)
    silver["LapTime_seconds"] = df["LapTime"].dt.total_seconds().astype("float64")

    silver["Compound"] = df["Compound"].astype("string")
    silver["TyreLife"] = df["TyreLife"].astype("int64")
    silver["Stint"] = df["Stint"].astype("int64")

    silver["TrackStatus"] = df["TrackStatus"].astype("string")

    return silver


In [15]:
test_round = rounds[0]

bronze_path = test_round / "session=R" / "data.parquet"
df_bronze = pd.read_parquet(bronze_path)

df_silver_test = transform_bronze_laps_to_silver(df_bronze)

df_silver_test.dtypes


season                      int64
event_round                 int64
session_type       string[python]
Driver             string[python]
LapNumber                   int64
LapTime_seconds           float64
Compound           string[python]
TyreLife                    int64
Stint                       int64
TrackStatus        string[python]
dtype: object

In [16]:
silver_round_path = SILVER_LAPS_BASE / test_round.name / "session=R"
silver_round_path.mkdir(parents=True, exist_ok=True)

df_silver_test.to_parquet(
    silver_round_path / "data.parquet",
    index=False
)


In [17]:
for round_path in rounds:
    bronze_file = round_path / "session=R" / "data.parquet"

    df_bronze = pd.read_parquet(bronze_file)
    df_silver = transform_bronze_laps_to_silver(df_bronze)

    silver_out = SILVER_LAPS_BASE / round_path.name / "session=R"
    silver_out.mkdir(parents=True, exist_ok=True)

    df_silver.to_parquet(
        silver_out / "data.parquet",
        index=False
    )


In [18]:
sample_files = list(SILVER_LAPS_BASE.rglob("data.parquet"))[:3]

[pd.read_parquet(f).dtypes for f in sample_files]


[season                      int64
 event_round                 int64
 session_type       string[python]
 Driver             string[python]
 LapNumber                   int64
 LapTime_seconds           float64
 Compound           string[python]
 TyreLife                    int64
 Stint                       int64
 TrackStatus        string[python]
 dtype: object,
 season                      int64
 event_round                 int64
 session_type       string[python]
 Driver             string[python]
 LapNumber                   int64
 LapTime_seconds           float64
 Compound           string[python]
 TyreLife                    int64
 Stint                       int64
 TrackStatus        string[python]
 dtype: object,
 season                      int64
 event_round                 int64
 session_type       string[python]
 Driver             string[python]
 LapNumber                   int64
 LapTime_seconds           float64
 Compound           string[python]
 TyreLife              

In [19]:
sum(pd.read_parquet(f).shape[0] for f in sample_files)


3134